# ICA apply and post-ICA autoreject

Apply the selection from 03 (manual if there is one, otherwise automatic) to
the epochs from 02 and record which components were removed. The ICA is not
refitted. Then run a second autoreject pass on the cleaned epochs, with the
settings in `[interpolate]` (settings.toml), and interpolate the channels
marked bad in 01. The first autoreject pass, in 02, selected the epochs for the
ICA fit.

The cleaned epochs are saved (`clean_ica-epo.fif`) and read back before
autoreject. A second copy of the final epochs, low-passed at 40 Hz, is saved
for other analyses (`interpolated_lp40-epo.fif`).

In [ ]:
subject_id = "101"
# set by the runner (papermill); None means: find settings.toml from cwd
config_path = None
# set by the runner, e.g. a test folder for derivatives_root
path_overrides = None

In [ ]:
import json
from datetime import datetime
from functools import partial
from html import escape

import autoreject
import mne
import pandas as pd
from IPython.display import display

from spectral.epochs import autoreject_overview
from spectral.helpers import load_or_create_report, save_subject_report
from spectral.ica import (apply_ica_decision, describe_ica_application,
                          read_ica_application, write_ica_application)
from spectral.utils import ProjectPaths, print_timestamp
from spectral.viz import plot_epochs

paths = ProjectPaths(subject_id, config_path=config_path, path_overrides=path_overrides)
subject_id = paths.subject_id  # zero-padded, as in the file names from 01
paths.create_directories()
print_timestamp("Setting up project paths")
paths.show()

# One report per subject: 01 creates it, each step adds its own section.
REPORT_SECTION = "04 ICA apply and interpolate"
report = load_or_create_report(paths, subject_id)
add_fig = partial(report.add_figure, section=REPORT_SECTION, replace=True)

mne.viz.set_browser_backend("matplotlib")
mne.set_config("MNE_BROWSER_THEME", "light")

## Apply the ICA selection

In [ ]:
epochs_source = paths.epochs / f"sub-{subject_id}_good_epochs-epo.fif"
epochs_good = mne.read_epochs(epochs_source, preload=True)
ica = mne.preprocessing.read_ica(paths.analysis / f"sub-{subject_id}_my_ica_model-ica.fif")
decision_path = paths.analysis / f"sub-{subject_id}_ica-decision.json"
decision = json.loads(decision_path.read_text())
application_path = paths.analysis / f"sub-{subject_id}_ica-application.json"

In [ ]:
# The application record lists the components that were removed. A manual
# selection made on an earlier fit is matched by topography
# ([cleaning] ica_corr_threshold).
epochs_clean, application = apply_ica_decision(
    epochs_good, ica, decision,
    corr_threshold=paths.config["cleaning"]["ica_corr_threshold"],
    input_source=epochs_source,
)
saved_epoch_files = epochs_clean.save(
    paths.analysis / f"sub-{subject_id}_clean_ica-epo.fif", overwrite=True)
application = write_ica_application(
    application_path, application, epochs_files=saved_epoch_files)
summary = describe_ica_application(application)
print(summary)
report.add_html(f"<p>{escape(summary)}</p>", title="ICA application",
                section=REPORT_SECTION, replace=True)

In [ ]:
# Average before and after ICA.
overlay = ica.plot_overlay(epochs_good.average(), show=False)
add_fig(overlay, title="ICA overlay, before and after", tags=("ica",))
overlay

In [ ]:
epochs_plot, epochs_timeseries, epochs_plot_psd = plot_epochs(
    epochs_clean, figures_path=paths.figures, subject=subject_id
)
add_fig(epochs_plot, title="Epochs after ICA", tags=("epochs",))
add_fig(epochs_plot_psd, title="Epochs after ICA, PSD", tags=("epochs", "psd"))

## Post-ICA autoreject

In [ ]:
# Read the cleaned epochs back from the file. MNE saves 32-bit data, so
# autoreject works on exactly what is in clean_ica-epo.fif, as it did when this
# was a separate notebook. The application record is checked against the file
# (all FIF parts).
clean_path = paths.analysis / f"sub-{subject_id}_clean_ica-epo.fif"
application = read_ica_application(
    paths.analysis / f"sub-{subject_id}_ica-application.json",
    epochs_path=clean_path)
epochs_clean = mne.read_epochs(clean_path, preload=True)

In [ ]:
_ip = paths.config["interpolate"]
ar = autoreject.AutoReject(
    n_interpolate=_ip["autoreject_n_interpolate"],
    n_jobs=-1,
    random_state=_ip["autoreject_random_state"],
    thresh_method=_ip["autoreject_thresh_method"],
    verbose=False,
).fit(epochs_clean)
epochs_ar, reject_log = ar.transform(epochs_clean, return_log=True)

# If fewer than min_epochs_after_ar epochs are left, keep the epochs from
# before this pass and add a note to the report.
_min_epochs = _ip["min_epochs_after_ar"]
ar_applied = len(epochs_ar) >= _min_epochs
if not ar_applied:
    print(f"WARNING autoreject kept {len(epochs_ar)} of {len(epochs_clean)} epochs "
          f"(< {_min_epochs}). Keeping the un-rejected epochs instead.")
    report.add_html(
        html=f"<p><b>Autoreject skipped.</b> The post-ICA pass kept only "
             f"{len(epochs_ar)} of {len(epochs_clean)} epochs, below the "
             f"min_epochs_after_ar = {_min_epochs} floor, so its rejection was "
             f"discarded and the un-rejected epochs carried forward. "
             f"This subject needs review.</p>",
        title="Autoreject skipped", section=REPORT_SECTION, replace=True)
    epochs_ar = epochs_clean.copy()
else:
    print(f"autoreject kept {len(epochs_ar)} of {len(epochs_clean)} epochs")

In [ ]:
# Left: rejected and interpolated epochs per channel. Right: channels with the
# most rejections. A channel that is high here for many subjects points to a
# problem with that electrode.
ar_table, ar_fig = autoreject_overview(
    reject_log, epochs_clean, epochs_ar, ar=ar,
    interpolated_channels=epochs_clean.info["bads"],
)
display(ar_table)
add_fig(ar_fig, title="Autoreject, rejections per channel",
        tags=("autoreject",))

## Interpolate and save

`interpolate_bads` interpolates the channels marked bad in 01, in all epochs.
This is separate from the per-epoch interpolation done by autoreject above.

In [ ]:
print(f"interpolating {len(epochs_ar.info['bads'])} channel(s): "
      f"{epochs_ar.info['bads'] or 'none'}")
epochs_interpolated = epochs_ar.copy().interpolate_bads(exclude=["VREF"])
epochs_interpolated.save(
    paths.analysis / f"sub-{subject_id}_interpolated-epo.fif", overwrite=True)

epochs_plot, epochs_timeseries, epochs_plot_psd = plot_epochs(
    epochs_interpolated, paths.figures, subject_id, stage="final_interpolated_epochs")
add_fig(epochs_plot, title="Final epochs", tags=("epochs",))
add_fig(epochs_plot_psd, title="Final epochs, PSD", tags=("epochs", "psd"))

## Low-passed copy (40 Hz)

The same final epochs, low-passed at 40 Hz, for analyses other than specparam
(`interpolated_lp40-epo.fif`). 05 still reads the unfiltered file above. The
filter comes after the ICA because ICLabel (03) needs the 1-100 Hz data.

In [ ]:
lowpass_hz = 40.0

# The recording from 01 is filtered and cut into the same epochs as in 02, so
# the filter does not distort the epoch edges. detrend=1 as in 02
# (create_epochs); the epoch files do not store it.
raw_lowpass = mne.io.read_raw_fif(
    paths.preprocessed / f"sub-{subject_id}_annotated_filtered_raw.fif", preload=True
).filter(l_freq=None, h_freq=lowpass_hz, picks=["eeg", "ecg"])
epochs_lowpass = mne.Epochs(
    raw_lowpass, epochs_good.events, tmin=epochs_good.tmin, tmax=epochs_good.tmax,
    baseline=None, detrend=1, preload=True, reject_by_annotation=False)
assert len(epochs_lowpass) == len(epochs_good), "low-passed epochs do not match 02"

# Same components removed, same autoreject decisions as above (made on the
# 1-100 Hz data), same channels interpolated.
epochs_lowpass, _ = apply_ica_decision(
    epochs_lowpass, ica, decision,
    corr_threshold=paths.config["cleaning"]["ica_corr_threshold"])
if ar_applied:
    epochs_lowpass = ar.transform(epochs_lowpass, reject_log=reject_log)
epochs_lowpass.interpolate_bads(exclude=["VREF"])
epochs_lowpass.save(
    paths.analysis / f"sub-{subject_id}_interpolated_lp{lowpass_hz:g}-epo.fif",
    overwrite=True)
print(f"saved {len(epochs_lowpass)} epochs low-passed at {lowpass_hz:g} Hz")

In [ ]:
# ICA counts are taken from the application record, so they describe the data
# as it was cleaned, even if the decision file is edited later.
_excluded = application["excluded_components"]
metadata_df = pd.DataFrame({
    "subject_id": [subject_id],
    "timestamp": [datetime.now()],
    "nr_interpolated_channels": [len(epochs_ar.info["bads"])],
    "nr_dropped_ica": [len(_excluded)],
    "nr_retained_ica": [application["n_components"] - len(_excluded)],
    "total_ica_components": [application["n_components"]],
    "nr_epochs_after_ar": [len(epochs_ar)],
    "autoreject_applied": [ar_applied],   # False: epochs from before this pass were kept; check this subject
    "bad_channels": [",".join(epochs_ar.info["bads"])],
    "exclusion_source": [application["exclusion_source"]],
    "ica_fingerprint": [application["ica_fingerprint"]],
})
metadata_path = paths.specparam / f"sub-{subject_id}_ica_metadata.csv"
metadata_df.to_csv(metadata_path, index=False)
print(f"Metadata saved to: {metadata_path}")
display(metadata_df.T)

In [ ]:
save_subject_report(report, paths, subject_id)